# 03. Preprocessing

Encodes student IDs and course modules into zero-indexed numeric categories and performs leave-one-out split.

In [1]:
import os
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder

registration_path = os.path.join("..", "..", "datasets", "raw", "studentRegistration.csv")
df = pd.read_csv(registration_path)

# Drop missing date_registration as in pipeline script
df = df.dropna(subset=["date_registration"])

user_enc = LabelEncoder()
item_enc = LabelEncoder()

df['user_idx'] = user_enc.fit_transform(df['id_student'])
df['item_idx'] = item_enc.fit_transform(df['code_module'])

# Save Encoders to artifacts and saved_models
artifacts_dir = os.path.join("..", "..", "artifacts", "recommendation")
saved_models_dir = os.path.join("..", "..", "saved_models", "recommendation")

os.makedirs(artifacts_dir, exist_ok=True)
os.makedirs(saved_models_dir, exist_ok=True)

joblib.dump(user_enc, os.path.join(artifacts_dir, "user_encoder.joblib"))
joblib.dump(item_enc, os.path.join(artifacts_dir, "item_encoder.joblib"))
joblib.dump(user_enc, os.path.join(saved_models_dir, "user_encoder.joblib"))
joblib.dump(item_enc, os.path.join(saved_models_dir, "item_encoder.joblib"))

# Leave-one-out Split by date_registration
df = df.sort_values(by=["user_idx", "date_registration"])
df["rank"] = df.groupby("user_idx").cumcount(ascending=False)

train_df = df[df["rank"] > 1]
val_df = df[df["rank"] == 1]
test_df = df[df["rank"] == 0]

processed_dir = os.path.join("..", "..", "datasets", "processed")
os.makedirs(processed_dir, exist_ok=True)

train_df.to_csv(os.path.join(processed_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(processed_dir, "val.csv"), index=False)
test_df.to_csv(os.path.join(processed_dir, "test.csv"), index=False)
df.to_csv(os.path.join(processed_dir, "recommendation_processed.csv"), index=False)

print(f"Preprocessing complete. Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

Preprocessing complete. Train size: 268, Val size: 3525, Test size: 28755
